# **<p align="center"> Fragalysis Data Dowload and Consolidation </p>**

## <ins> Description </ins>
### Dowloaded different data type folders (`zips`), consolidated data into one unique folder without duplicates (`concat`), and transfered data to Wilson Cluster for Analysis 

<details >
<summary> <font size="6"> <ins>Data Extraction Protocol </ins> </font> </summary>

- Went into fragalysis lb32627-66 website: [link](https://fragalysis.diamond.ac.uk/viewer/react/preview/target/A71EV2A/tas/lb32627-66)

- Downloaded each individual data type: **2fofc** | **fofc** | **event-maps** | **mtz** | **pdb** | **sdf** 

- Each Data Folder Type contained some repeated datasets.

- Added the extra data files to a "base" data folder

- Selected the event maps to be the base

    - This contained : `aligned_files` folder | `extra_files_1` folder | `scripts` folder | `yaml_files` folder | `errors.csv` | `metadata.csv` |  `README.pdf` | `smiles.smi`

- Dowloaded sdf folder and added: `{dataset}_combined.sdf`

- Downloaded pdb folder, created `crystallographic_files` folder, added: `crystallographic_files/{dataset}/{datasubset}.pdb`

- Downloaded mtz folder, added: `crystallographic_files/{dataset}/{datasubset}.mtz`

- Downloaded 2fo-fc folder, added:  `{datasubset}_sigmaa.ccp4` | `{datasubset}_sigmaa_crystallographic.ccp4`

- Downloaded fo-fc folder, added: `{datasubset}_diff.ccp4` | `{datasubset}_diff_crystallographic.ccp4`
</details>

</br>

<details >
<summary> <font size = "6"> <ins>Concat Dir Structure </ins> </font> </summary>

- `crystallographic_files`

    Refers to the data collected for one specific crystal with one protein soaked with one specific compound (fragment of ligand).

    Experimentally several repeats can be performed for the same protein-fragment couple. Each experiment has one unique datasetID for the crystal and one unique molecule-BatchID for the compound. The later is not found in the metadata.

    My assumption is that in fragalysis, there is no duplicates of protein-compound pairs, and the best of the repeats is chosen to be placed.

- `aligned_files`

    Refers to the crystallographic data which was later aligned, so that each different dataset files contain the most relatable coordinates center amongst them.

    In addition, each dataset - correspondent to one protein-compound pair - is subdivided into dataset subsets - correspondent to different locations where the compound binds in the protein surface. 

- `metadata.csv`

    Contains the metadata related to manually curated labels associated to datasets and compound ids, which identify protein binding sites, compound chemical series, and others.

- `README.pdb` 

    Goes over the key organisation of the fragalysis data, like here described.

</details>

</br>

<details>
<summary> <font size = "6"> <ins> Key Data Types </ins> </font> </summary>

- **pdb** | **pandda-eventmap** | **mtz** | **2fo-fc** | **fo-fc** | **sdf** | 
**cif** | 
</details>

</br>

<details>
<summary> <font size = "6">  <ins> Key Extra Data Structures for each Data Type Folder </ins> </font> </summary>

Extra files found in other folders that not the base folder, which were added to concat:

- Base: event map

- Sdf: **{dataset}_combined.sdf**

- pdb: **crystallographic_files/{dataset}/{datasubset}.pdb**

- mtz: **crystallographic_files/{dataset}/{datasubset}.mtz**

- 2fo-fc: **{datasubset}_sigmaa.ccp4** and 
**{datasubset}_sigmaa_crystallographic.ccp4** 

- fo-fc: **{datasubset}_diff.ccp4** and 
**{datasubset}_diff_crystallographic.ccp4** 
</details>

</br>

<details open>
<summary> <font size = "6"> <ins> Open Questions </ins> </font></summary>

- Does Fragalysis Provide One Crystal per Protein-Fragment per or are there duplicates associated to different compound solution and protein crystal batches?

</details>

In [ ]:
# Set Root and Concat Dir variables
from pathlib import Path, PureWindowsPath

repoDir = Path("../../../..")
fragDir = repoDir.joinpath( PureWindowsPath(r".\data\ev2a\fragalysis").as_posix())  # Fragalysis Folder
concatDir = fragDir.joinpath( r"01-concatDir")
concatDir.mkdir( parents = True, exist_ok = True )
referenceID = "A71EV2A" # Protein Name

../../../../data/ev2a/fragalysis/01-concatDir


In [ ]:
# Cp fragDir/map/* > concatdir/ 
from pathlib import Path
from shutil import copy2

def iterFileSys(srcDir:Path, relPath: str = "."):
    for item in srcDir.iterdir():
        if item.is_file():
            copy2( item, concatDir.joinpath( relPath).joinpath( item.name) )
        elif item.is_dir():
            newPath = relPath +"/"+ item.name
            concatDir.joinpath( newPath).resolve().mkdir(parents=True,
                                                         exist_ok= True)
            iterFileSys( item, relPath = newPath )

iterFileSys( fragDir.joinpath( "map") )

In [ ]:
# Add sdf extra files into it
from shutil import copy2

copy2( fragDir.joinpath("sdf/{}_combined.sdf".format(referenceID)),
       concatDir.joinpath("{}_combined.sdf".format(referenceID) ) )

WindowsPath('D:/xaidar/data/my_fragalysis/ev2a/concatDir/A71EV2A_combined.sdf')

In [ ]:
# Add pdb extra files into it
from shutil import copytree

copytree( src = fragDir.joinpath( "pdb/crystallographic_files"),
          dst = concatDir.joinpath("crystallographic_files") )

WindowsPath('D:/xaidar/data/my_fragalysis/ev2a/concatDir/crystallographic_files')

In [ ]:
# Add mtz extra files into it
from shutil import copy2

for dataset in fragDir.joinpath( "mtz/crystallographic_files").iterdir():
    crystalDataDir = concatDir.joinpath( "crystallographic_files/" \
                                        "{}".format(dataset.name) )
    crystalDataDir.mkdir( parents=True, exist_ok=True)
    for file in dataset.iterdir():
        newFile = crystalDataDir.joinpath( file.name )
        copy2( file, newFile)

In [ ]:
# Add 2fo-fc extra files into concat Dir
from shutil import copy2

for dataset in fragDir.joinpath( "2fofc/aligned_files").iterdir():
    newDatasetDir = concatDir.joinpath( "aligned_files/{}".format(dataset.name) )
    assert newDatasetDir.exists(), ("This"
        " dataset directory does not exist in: "
        "{}").format(newDatasetDir.as_posix() ) 

    fileNameList = [ "{}{}".format(dataset.name, fileName) for fileName in 
                 ["_sigmaa.ccp4","_sigmaa_crystallographic.ccp4" ] ]
    for fileName in fileNameList :
        file = dataset.joinpath( fileName)
        assert file.exists(), "This file does not exist in rootdir/2fofc: "\
            "{}".format(file)
        newFile = newDatasetDir.joinpath( fileName )
        copy2( file, newFile)

In [ ]:
# Add fo-fc extra files into concat Dir
from shutil import copy2

for dataset in fragDir.joinpath( "fofc/aligned_files").iterdir():
    newDatasetDir = concatDir.joinpath( "aligned_files/" \
                                        "{}".format(dataset.name) )
    assert newDatasetDir.exists(), ("This"
        " dataset directory does not exist in: "
        "{}").format(newDatasetDir.as_posix() ) 

    fileNameList = [ "{}{}".format(dataset.name, fileName) for fileName in 
                 ["_diff.ccp4","_diff_crystallographic.ccp4" ] ]
    for fileName in fileNameList :
        file = dataset.joinpath( fileName)
        assert file.exists(), "This file does not exist in rootdir/fofc: "\
            "{}".format(file)
        newFile = newDatasetDir.joinpath( fileName )
        copy2( file, newFile)